# Baseline BioBERTpt — RECLin-PT (Colab T4)

Treina o baseline de extração de relação em 3 classes (`negation_of`, `associated_with`, `no_relation`) sobre os splits do SemClinBr e reporta o **F1 da classe `negation_of`**.

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre o secret `GITHUB_PAT` (ícone de chave 🔑 na barra lateral).

O treino salva um **checkpoint por época no Google Drive** e **retoma automaticamente** se o runtime cair — basta rodar tudo de novo.

## 1. Clonar o repositório

In [ ]:
import os, subprocess
from google.colab import userdata

TOKEN = userdata.get('GITHUB_PAT')           # secret do Colab, nunca hard-coded
REPO_NAME = 'RECLin-PT-Min'                   # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

url = f'https://{TOKEN}@github.com/angeloalsf/{REPO_NAME}'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', url, REPO], check=True)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)

subprocess.run(['git', '-C', REPO, 'config', 'user.name', GIT_USER_NAME], check=True)
subprocess.run(['git', '-C', REPO, 'config', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', url], check=True)
print('Repo pronto em', REPO)


## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'


## 3. Montar o Google Drive (checkpoints)

Os checkpoints por época vão para `MyDrive/RECLin-PT-Min/checkpoints/`. Como ficam no Drive, sobrevivem à desconexão do Colab e permitem retomar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Pasta de checkpoints no Drive: sobrevive a quedas do runtime.
CKPT_DIR = f'/content/drive/MyDrive/{REPO_NAME}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints em:', CKPT_DIR)
existe = os.path.isfile(os.path.join(CKPT_DIR, 'checkpoint.pt'))
print('Checkpoint existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')


## 4. Instalar dependências

O Colab já traz `torch` com CUDA — **não reinstalamos** para não quebrar a GPU. Instalamos só o que falta.

In [ ]:
!pip install -q "transformers>=4.40" "scikit-learn>=1.4" lxml


## 5. Conferir os splits

Os splits (`data/splits/*.jsonl`) são versionados no repo, então já vêm com o clone. Se faltarem e os XMLs estiverem presentes, geramos na hora.

In [ ]:
import subprocess, sys
splits = os.path.join(REPO, 'data', 'splits')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    xml = os.path.join(REPO, 'SemClinBr-xml-public-v1')
    if os.path.isdir(xml) and os.listdir(xml):
        print('Splits ausentes; gerando a partir dos XMLs...')
        subprocess.run([sys.executable, 'src/parse_semclinbr.py',
                        '--xml-dir', 'SemClinBr-xml-public-v1',
                        '--out', 'data/processed/dataset.jsonl'], cwd=REPO, check=True)
        subprocess.run([sys.executable, 'src/make_splits.py'], cwd=REPO, check=True)
    else:
        raise FileNotFoundError(
            'Splits não encontrados e XMLs ausentes. Faça commit de data/splits/ '
            'no repo, ou suba a pasta SemClinBr-xml-public-v1/ para o Colab.')

for f in need:
    n = sum(1 for _ in open(os.path.join(splits, f), encoding='utf-8'))
    print(f'{f}: {n} docs')


## 6. Treinar o baseline BioBERTpt (com retomada)

Encoder `pucpr/biobertpt-all`, marcadores de entidade, 3 classes, `class_weight=balanced`. Melhor época escolhida pelo macro-F1 no **dev**; **test** reportado uma única vez.

`--ckpt-dir` aponta para o Drive: ao fim de cada época grava `checkpoint.pt`. Se o runtime cair, **rode esta célula de novo** (depois das de cima) e ele **pula as épocas já feitas** e continua. Se já tiver feito todas, vai direto para a avaliação final.

Na T4, ~algumas dezenas de minutos por época (~378k pares no train).

In [ ]:
%cd $REPO
!python src/baseline_biobertpt.py \
    --splits-dir data/splits \
    --model pucpr/biobertpt-all \
    --epochs 3 \
    --batch-size 32 \
    --max-gap 20 \
    --max-length 192 \
    --seed 42 \
    --ckpt-dir "$CKPT_DIR" \
    --out results/baseline_biobertpt.json


## 7. Ver os resultados

In [ ]:
import json
r = json.load(open(os.path.join(REPO, 'results', 'baseline_biobertpt.json'), encoding='utf-8'))
print('Candidatos:', r['n_candidates'])
print('Macro-F1:', round(r['test_macro_f1'], 4))
print('Micro-F1:', round(r['test_micro_f1'], 4))
print('\nF1 por classe:')
for k, v in r['test_f1_per_class'].items():
    mark = '   <<< NEGAÇÃO' if k == 'negation_of' else ''
    print(f'  {k:<18s} {v:.4f}{mark}')


## 8. (Opcional) Salvar os resultados no GitHub

`results/` é gitignored, então forçamos o `add` do JSON de métricas.

In [ ]:
import subprocess
subprocess.run(['git', '-C', REPO, 'add', '-f',
                'results/baseline_biobertpt.json'], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m',
                'resultados: baseline BioBERTpt (Colab T4)'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('Push concluído (se houve mudanças).')
